# EDA Data Harga Pangan — Propagasi Wilayah (Prasyarat iTransformer)

Notebook ini menganalisis **konsumen** (`konsumen_raw.csv`, 6 provinsi) dan **produsen** (`produsen_raw.csv`, 5 provinsi), periode **2021–2025**.

**Tujuan:** mendeteksi pola propagasi harga antar wilayah, stasioneritas, korelasi spasial, lag produsen→konsumen, spread rantai pasok, dan anomali — sebagai dasar desain fitur/preprocessing untuk **iTransformer**.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import seaborn as sns

try:
    import missingno as msno
    HAS_MSNO = True
except ImportError:
    HAS_MSNO = False

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 12

# ── Path data (cwd = akar proyek saat notebook dijalankan dari folder ini)
for candidate in [Path(".").resolve(), Path(".").resolve().parent]:
    _kf = candidate / "Dataset" / "processed" / "konsumen_raw.csv"
    if _kf.exists():
        ROOT = candidate
        break
else:
    ROOT = Path(".").resolve()

DATA_DIR = ROOT / "Dataset" / "processed"
FILE_K = DATA_DIR / "konsumen_raw.csv"
FILE_P = DATA_DIR / "produsen_raw.csv"

PROV_KONSUMEN = ["dki", "jabar", "jateng", "jatim", "sulsel", "sumut"]
PROV_PRODUSEN = ["jabar", "jateng", "jatim", "sulsel", "sumut"]
# Provinsi dengan kedua sisi (tanpa DKI pada produsen)
COMMON_PROV_SPREAD = ["jabar", "jateng", "jatim", "sulsel", "sumut"]

KOMODITAS = ["beras", "bawang", "cabai"]

print("ROOT =", ROOT)
print("Files exist:", FILE_K.exists(), FILE_P.exists())


In [ ]:
def load_panel(path: Path, prefix: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)
    df.attrs["prefix"] = prefix
    return df


df_k = load_panel(FILE_K, "konsumen")
df_p = load_panel(FILE_P, "produsen")

num_cols_k = [c for c in df_k.columns if c != "date"]
num_cols_p = [c for c in df_p.columns if c != "date"]

for c in num_cols_k:
    df_k[c] = pd.to_numeric(df_k[c], errors="coerce")
for c in num_cols_p:
    df_p[c] = pd.to_numeric(df_p[c], errors="coerce")

df_k.head()


## 1. Kelangsungan Data & Missing Values

Memetakan **NaN per kolom** dan pola spasio-temporal gap (akhir pekan vs rangkaian panjang). Kolom fokus: `beras_sulsel` (konsumen) dan `beras_jatim` (produsen).

In [ ]:
def pct_nan(frame: pd.DataFrame, cols=None):
    cols = cols or [c for c in frame.columns if c != "date"]
    s = frame[cols].isna().mean().sort_values(ascending=False) * 100
    return s.rename("pct_nan")


nan_k = pct_nan(df_k, num_cols_k)
nan_p = pct_nan(df_p, num_cols_p)

display_k = pd.DataFrame({"konsumen": nan_k}).join(pd.DataFrame({"produsen": nan_p}), how="outer")
print("Persentase NaN per kolom (%) — ringkas:")
print("--- KONSUMEN (top 10) ---")
print(nan_k.head(10).to_string())
print("\n--- PRODUSEN (top 10) ---")
print(nan_p.head(10).to_string())


In [ ]:
# Heatmap pola missing (missingno jika ada; fallback seaborn)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if HAS_MSNO:
    msno.matrix(df_k.set_index("date")[num_cols_k], ax=axes[0], sparkline=False)
    axes[0].set_title("Missing matrix — Konsumen")
    msno.matrix(df_p.set_index("date")[num_cols_p], ax=axes[1], sparkline=False)
    axes[1].set_title("Missing matrix — Produsen")
else:
    sns.heatmap(df_k[num_cols_k].isna(), cbar=False, ax=axes[0], yticklabels=False)
    axes[0].set_title("Missing (hitam=NaN) — Konsumen")
    sns.heatmap(df_p[num_cols_p].isna(), cbar=False, ax=axes[1], yticklabels=False)
    axes[1].set_title("Missing (hitam=NaN) — Produsen")

plt.tight_layout()
plt.show()

print(
    "Insight iTransformer: pola missing berulang (mis. akhir pekan) menjadikan panjang deret tidak seragam; "
    "input sequence perlu **reindex harian + mask/imputasi** konsisten agar attention tidak menafsirkan gap sebagai sinyal."
)


In [ ]:
def analyze_gap_pattern(df: pd.DataFrame, col: str, label: str):
    m = df[col].isna()
    sub = df.loc[m, "date"].copy()
    if sub.empty:
        print(f"{label}: tidak ada NaN pada {col}")
        return
    sub = pd.DataFrame({"date": sub})
    sub["dow"] = sub["date"].dt.dayofweek  # 0=Senin … 5=Sabtu 6=Minggu
    dow_counts = sub["dow"].value_counts().sort_index()
    names = ["Sen", "Sel", "Rab", "Kam", "Jum", "Sab", "Min"]
    print(f"\n=== {label} | {col} ===")
    print("Distribusi NaN per hari dalam minggu:")
    for i, n in enumerate(names):
        print(f"  {n}: {int(dow_counts.get(i, 0))}")

    # rangkaian berturut-turut NaN
    is_na = df[col].isna().astype(int)
    grp = (is_na != is_na.shift()).cumsum()
    runs = df.groupby(grp).agg(
        start=("date", "first"),
        len_na=(col, lambda s: s.isna().all()),
        length=(col, "size"),
    )
    runs = runs[runs["len_na"]]
    runs = runs.sort_values("length", ascending=False)
    print("Run NaN terpanjang (hari berturut-turut):", int(runs["length"].max()) if len(runs) else 0)
    print(runs.head(5).to_string())


analyze_gap_pattern(df_k, "beras_sulsel", "Konsumen")
analyze_gap_pattern(df_p, "beras_jatim", "Produsen")

print(
    "\nInsight iTransformer: jika dominan **Sab–Min**, interpretasi sebagai efek pelaporan/pasar tutup; "
    "model dengan window harian harus memutuskan apakah **forward-fill harga Jumat** atau **mask token** lebih sesuai."
)


## 2. Karakteristik Deret Waktu

Tren harian **Beras, Bawang, Cabai** semua provinsi; dekomposisi musiman pada wilayah kunci produsen; **uji ADF** per kolom.

In [ ]:
def commodity_cols(df: pd.DataFrame, commodity: str):
    return sorted([c for c in df.columns if c.startswith(f"{commodity}_")])


def plot_commodity_all_provinces(df: pd.DataFrame, commodity: str, title_suffix: str):
    cols = commodity_cols(df, commodity)
    fig, ax = plt.subplots(figsize=(13, 5))
    for c in cols:
        ax.plot(df["date"], df[c], label=c.replace(f"{commodity}_", ""), linewidth=1.1, alpha=0.85)
    ax.set_title(f"{commodity.upper()} — semua provinsi ({title_suffix})")
    ax.set_xlabel("Tanggal")
    ax.set_ylabel("Harga")
    ax.legend(ncol=6, fontsize=8, loc="upper left")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.xticks(rotation=35)
    plt.tight_layout()
    plt.show()


plot_commodity_all_provinces(df_k, "beras", "konsumen")
print(
    "Insight iTransformer: divergensi antarwilayah pada shock menunjukkan **channel variabel per-node**; "
    "iTransformer dapat memisahkan representasi antar variabel jika scaling provinsi-seimbang."
)

plot_commodity_all_provinces(df_k, "bawang", "konsumen")
print("Insight: volatilitas bawang tinggi → residual/noise lebih besar; pertimbangkan transformasi stabil (log-return).")

plot_commodity_all_provinces(df_k, "cabai", "konsumen")
print("Insight: cabai sering lonjak → attention bisa belajar **event spikes**; penting normalisasi robust.")

plot_commodity_all_provinces(df_p, "beras", "produsen")
print("Insight produsen beras: level lebih rendah dari konsumen — spread positif sistemik; cocok sebagai variabel pemicu.")

plot_commodity_all_provinces(df_p, "bawang", "produsen")
print("Insight: fluktuasi prod–kons serupa musiman; residual bisa mengandung propagasi panjang.")

plot_commodity_all_provinces(df_p, "cabai", "produsen")
print("Insight: shock cabai di produsen sering mendahului lonjakan konsumen — relevan untuk horizon prediksi pendek.")


In [ ]:
# Seasonal decomposition — produsen utama: Jawa Timur, komoditas Beras (series kontinu setelah interpolasi harian)
KEY_COL = "beras_jatim"
s_raw = df_p.set_index("date")[KEY_COL].asfreq("D")
s_filled = s_raw.interpolate(limit_direction="both")

period = 7  # bulanan kurang panjang untuk harian 2021–2025; gunakan weekly seasonality
dec = seasonal_decompose(s_filled, model="additive", period=period, extrapolate_trend="freq")

fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
dec.observed.plot(ax=axes[0], title="Observed", legend=False)
dec.trend.plot(ax=axes[1], title="Trend", legend=False)
dec.seasonal.plot(ax=axes[2], title="Seasonal (period=7)", legend=False)
dec.resid.plot(ax=axes[3], title="Residual", legend=False)
axes[0].set_ylabel(KEY_COL)
plt.suptitle(f"Decomposition produsen — {KEY_COL} (interp harian untuk kontinuitas)")
plt.tight_layout()
plt.show()

print(
    "Insight iTransformer: komponen musiman mingguan menunjukkan **periodic bias**; "
    "model bisa di-bantu differencing musiman atau membagi resid untuk prediksi shock propagasi."
)


In [ ]:
def adf_report(series: pd.Series, name: str):
    x = pd.to_numeric(series, errors="coerce").dropna()
    if len(x) < 20:
        return {"kolom": name, "n": len(x), "adf_stat": np.nan, "p_value": np.nan, "stasioner_5pct": np.nan}
    res = adfuller(x, autolag="AIC")
    return {
        "kolom": name,
        "n": len(x),
        "adf_stat": res[0],
        "p_value": res[1],
        "stasioner_5pct": res[1] < 0.05,
    }


rows = []
for c in num_cols_k:
    rows.append(adf_report(df_k[c], f"k:{c}"))
for c in num_cols_p:
    rows.append(adf_report(df_p[c], f"p:{c}"))

adf_df = pd.DataFrame(rows)
print("ADF test (H0: unit root). Ringkas:")
print(adf_df.groupby("stasioner_5pct").size())
display_cols = ["kolom", "n", "adf_stat", "p_value", "stasioner_5pct"]
print(adf_df.sort_values("p_value").head(15)[display_cols].to_string(index=False))

print(
    "\nInsight iTransformer: banyak kolom non-stasioner → **differencing / return** umumnya diperlukan sebelum scaling; "
    "patch length dan horizon harus diuji ulang setelah transformasi."
)


## 3. Korelasi Spasial antar Wilayah (Komoditas Sama)

Heatmap Pearson antar kolom provinsi untuk komoditas yang sama — analog struktur dependensi yang dicoba ditangkap attention.

In [ ]:
def corr_heatmap(df: pd.DataFrame, commodity: str, title: str):
    cols = commodity_cols(df, commodity)
    if len(cols) < 2:
        print(f"Tidak cukup kolom untuk {commodity}")
        return
    cm = df[cols].corr()
    labels = [c.replace(f"{commodity}_", "").upper() for c in cols]
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.tight_layout()
    plt.show()
    # pasangan terkuat (off-diagonal)
    stack = cm.where(np.triu(np.ones(cm.shape), k=1).astype(bool)).stack()
    if len(stack):
        top = stack.abs().sort_values(ascending=False).head(5)
        print(f"Top korelasi |{commodity}|:\n", top)


corr_heatmap(df_k, "beras", "Korelasi antarwilayah — BERAS (konsumen)")
print(
    "Insight iTransformer: blok korelasi tinggi ⇒ **grup pasar terintegrasi**; "
    "attention bisa redundant — pertimbangkan pooling atau regularisasi graf."
)

corr_heatmap(df_p, "beras", "Korelasi antarwilayah — BERAS (produsen)")
print(
    "Insight: produsen Jawa sering bergerak bersama → kemungkinan **sender dominan** di Jawa untuk merambat ke konsumen."
)

corr_heatmap(df_k, "bawang", "Korelasi — BAWANG (konsumen)")
corr_heatmap(df_k, "cabai", "Korelasi — CABAI (konsumen)")

# Lintas panel prod–kons (komoditas sama, aligned tanggal)
cross_pairs = [
    ("beras_jatim", "beras_dki", "Prod JATIM beras ↔ Kons DKI beras"),
    ("beras_jateng", "beras_jabar", "Prod JATENG beras ↔ Kons JABAR beras"),
    ("cabai_jatim", "cabai_dki", "Prod JATIM cabai ↔ Kons DKI cabai"),
]
rows_x = []
for pc, kc, label in cross_pairs:
    if pc not in df_p.columns or kc not in df_k.columns:
        continue
    m = pd.DataFrame({"p": df_p[pc], "k": df_k[kc], "date": df_k["date"]}).dropna()
    if len(m) < 30:
        continue
    r = m["p"].corr(m["k"])
    rows_x.append((label, r))
print("\nKorelasi prod–kons (aligned, Pearson):")
for lab, r in rows_x:
    print(f"  {lab}: r = {r:.4f}")
print(
    "\nInsight iTransformer: **JATIM prod vs DKI kons** mengukur hubungan **sender–receiver** geografis; "
    "nilai tinggi mendukung jalur propagasi yang dapat diprioritaskan sebagai pair supervision atau attention bias."
)


## 4. Propagasi & Analisis Lag (Produsen → Konsumen)

**Cross-correlation** antara \(P_t\) dan \(K_{t+h}\) untuk **h = 0 … 14**: produsen sebagai pemicu, konsumen sebagai respons.

In [ ]:
def lagged_crosscorr(prod: pd.Series, kons: pd.Series, max_lag: int = 14):
    # Lag h >= 0: corr(prod[t], kons[t+h]); prod mendahului konsumen jika puncak di h > 0.
    prod = pd.to_numeric(prod, errors="coerce")
    kons = pd.to_numeric(kons, errors="coerce")
    base = pd.DataFrame({"p": prod, "k": kons}).dropna()
    corrs = []
    for h in range(0, max_lag + 1):
        a = base["p"].iloc[:-h] if h > 0 else base["p"]
        b = base["k"].iloc[h:] if h > 0 else base["k"]
        n = min(len(a), len(b))
        if n < 30:
            corrs.append(np.nan)
            continue
        corrs.append(np.corrcoef(a.iloc[-n:], b.iloc[-n:])[0, 1])
    return np.array(corrs)


def plot_lag_curve(prod_series, kons_series, title: str, max_lag: int = 14):
    corrs = lagged_crosscorr(prod_series, kons_series, max_lag=max_lag)
    lags = np.arange(len(corrs))
    plt.figure(figsize=(10, 4))
    plt.bar(lags, corrs, color="steelblue", alpha=0.85)
    plt.xlabel("Lag h (hari): konsumen merespons setelah produsen")
    plt.ylabel("Korelasi Pearson")
    plt.title(title)
    peak = int(np.nanargmax(corrs)) if np.any(~np.isnan(corrs)) else 0
    plt.axvline(peak, color="crimson", linestyle="--", label=f"peak lag={peak}")
    plt.legend()
    plt.tight_layout()
    plt.show()
    return peak, corrs


# Pasangan kritis: prod JATIM beras vs kons DKI beras (sesuai contoh naming)
pk = df_p["beras_jatim"]
kk = df_k["beras_dki"]
peak, corrs = plot_lag_curve(pk, kk, "Lag corr: PROD beras_jatim → KONS beras_dki")
print(f"Puncak korelasi pada lag ≈ {peak} hari (estimasi delay propagasi).")

print(
    "\nInsight iTransformer: lag optimal memberi **prior causal spacing** untuk patch/time bias atau constrained attention; "
    "jika peak di lag kecil, hubungan hampir kontemporer (pasar efisien); lag besar ⇒ rantai distribusi panjang."
)

# Tambahan pasangan strategis (Jateng prod ↔ Jateng kons; Sumut prod ↔ Sumut kons)
pairs = [
    ("beras_jateng", "beras_jateng", "PROD vs KONS JATENG (beras)"),
    ("beras_sumut", "beras_sumut", "PROD vs KONS SUMUT (beras)"),
    ("cabai_jatim", "cabai_jatim", "PROD vs KONS JATIM (cabai)"),
]
for pc, kc, ttl in pairs:
    peak_i, _ = plot_lag_curve(df_p[pc], df_k[kc], ttl)
    print(f"  {ttl}: peak lag ≈ {peak_i}")


## 5. Analisis Spread (Konsumen − Produsen)

Untuk provinsi **Jabar, Jateng, Jatim, Sulsel, Sumut**: spread mengukur **margin/tekanan rantai pasok**.

In [ ]:
spread_frames = []
for prov in COMMON_PROV_SPREAD:
    for com in KOMODITAS:
        ck = f"{com}_{prov}"
        cp = f"{com}_{prov}"
        if ck not in df_k.columns or cp not in df_p.columns:
            continue
        spread_frames.append(
            pd.DataFrame(
                {
                    "date": df_k["date"],
                    "prov": prov.upper(),
                    "komoditas": com,
                    "spread": df_k[ck] - df_p[cp],
                }
            )
        )

spread_long = pd.concat(spread_frames, ignore_index=True)

fig, axes = plt.subplots(len(KOMODITAS), 1, figsize=(12, 10), sharex=True)
for i, com in enumerate(KOMODITAS):
    sub = spread_long[spread_long["komoditas"] == com]
    for prov in sub["prov"].unique():
        ssub = sub[sub["prov"] == prov]
        axes[i].plot(ssub["date"], ssub["spread"], label=prov, alpha=0.85)
    axes[i].set_ylabel(com)
    axes[i].legend(ncol=6, fontsize=8, loc="upper left")
    axes[i].set_title(f"Spread konsumen−produsen — {com}")

axes[-1].set_xlabel("Tanggal")
plt.tight_layout()
plt.show()

roll_vol = (
    spread_long.sort_values(["komoditas", "prov", "date"])
    .assign(
        roll_std=lambda d: d.groupby(["komoditas", "prov"])["spread"].transform(
            lambda s: s.rolling(14, min_periods=7).std()
        )
    )
)

plt.figure(figsize=(12, 4))
for com in KOMODITAS:
    sub = roll_vol[roll_vol["komoditas"] == com]
    plt.plot(sub.groupby("date")["roll_std"].mean(), label=com)
plt.title("Volatilitas spread agregat (rolling std 14 hari, rata provinsi)")
plt.legend()
plt.tight_layout()
plt.show()

print(
    "Insight iTransformer: spread non-stasioner dan lonjakan menandakan **structural break**; "
    "model multivariate harus mengkode provinsi dan komoditas sebagai token berbeda agar shock margin tidak tertukar."
)


## 6. Deteksi Anomali & Event Historis

Lonjakan dengan **Z-score rolling** dan band **moving standard deviation**; tanda vertikal **BBM Sept 2022** sebagai referensi.

In [ ]:
EVENT_DATES = [
    ("Kenaikan BBM Sept 2022", "2022-09-03"),
    ("Awal 2022", "2022-01-01"),
]


def flag_spikes_z(series: pd.Series, window: int = 30, z: float = 3.0):
    s = pd.to_numeric(series, errors="coerce")
    mu = s.rolling(window, min_periods=10).mean()
    sd = s.rolling(window, min_periods=10).std()
    zscore = (s - mu) / sd.replace(0, np.nan)
    return zscore.abs() > z, zscore


focus = df_k.set_index("date")["cabai_dki"].dropna()
spike_mask, zscores = flag_spikes_z(focus, window=30, z=3.0)
spike_mask = spike_mask.fillna(False)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(focus.index, focus.values, label="cabai_dki (konsumen)", color="black", linewidth=0.9)
spike_idx = focus.index[spike_mask.to_numpy(dtype=bool)]
spike_vals = focus.loc[spike_mask].values
ax.scatter(spike_idx, spike_vals, color="red", s=12, label="Z-score spike", zorder=5)

for name, ds in EVENT_DATES:
    ax.axvline(pd.Timestamp(ds), color="green", linestyle="--", alpha=0.7)
    ax.text(pd.Timestamp(ds), ax.get_ylim()[1], name, rotation=90, va="top", fontsize=8, color="green")

ax.set_title("Deteksi spike (contoh: cabai_dki) + marker referensi BBM 2022")
ax.legend()
plt.tight_layout()
plt.show()

# Moving std band untuk beras_jateng konsumen
w = 14
s2 = df_k.set_index("date")["beras_jateng"].dropna()
rm = s2.rolling(w).mean()
rs = s2.rolling(w).std()
plt.figure(figsize=(12, 4))
plt.plot(s2.index, s2.values, label="beras_jateng")
plt.fill_between(s2.index, rm - 2 * rs, rm + 2 * rs, alpha=0.25, label="±2σ rolling")
plt.axvline(pd.Timestamp("2022-09-03"), color="green", linestyle="--", alpha=0.8)
plt.title("Beras Jateng konsumen vs band volatilitas rolling")
plt.legend()
plt.tight_layout()
plt.show()

print(
    "Insight iTransformer: event makro menimbulkan **covariate shift**; "
    "latihan/validasi terbaik dengan split **time-based** dan segmentasi pra/pasca event atau augmentasi robust."
)


## Ringkasan untuk Pra-pemrosesan iTransformer

| Temuan | Implikasi praktis |
|--------|-------------------|
| Missing akhir pekan | Reindex harian, imputasi konsisten atau mask |
| ADF non-stasioner | Return/differencing per variabel |
| Korelasi spasial tinggi | Redundansi antar-node; scaling per provinsi |
| Lag prod-kons | Informasi prior jarak temporal antar channel |
| Spread & spike | Fitur ekologi rantai pasok; evaluasi out-of-time |

---
*Notebook ini dibuat untuk Gemastik Divisi III — Data Mining.*
